# Forecast Regret
How much does a weather forecast's error cost a transatlantic flight? For each **flight** we plan on a forecast source's winds, fly that **plan** through the **truth**, and compare it with the **hindsight plan**, the best candidate with the truth known.
**Regret** is the extra fuel, time and CO₂ that plan costs over the hindsight plan.

In [ ]:
import os, time
from pathlib import Path

from regret import offline
offline.install_if_requested()
_t0 = time.perf_counter()

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(_root)  # the kernel starts in notebooks/; config paths are relative to the repo root

import pandas as pd
import plotly.offline
from IPython.display import HTML, display

from regret import config
from regret.figures import showcase_map
from regret.figures.headline import FIG_ID as HEADLINE_ID, build_headline
from regret.figures.showcase_map import FIG_ID as MAP_ID, build_showcase_map
from regret.figures.lead_regret import FIG_ID as LEAD_ID, build_lead_regret
from regret.figures.wind_error import FIG_ID as WIND_ID, build_wind_error
from regret.figures.sanity import FIG_ID as SANITY_ID, build_sanity

cfg = config.get_config()
df = pd.read_parquet(cfg.results_path)
plotly.offline.init_notebook_mode(connected=False)  # plotly.js loaded once, inline
display(HTML(showcase_map.geo_assets_html()))  # basemap preloaded, so nothing is fetched from a CDN


def show(fig, fig_id):
    display(HTML(fig.to_html(full_html=False, include_plotlyjs=False, div_id=fig_id,
                                 config={"displayModeBar": False})))

## 1. Headline: what forecast error costs
Mean (with 90% CI), median and p90 regret in fuel, time and CO₂, by forecast source and lead, eastbound and westbound flights pooled.
On a narrow screen, scroll the table sideways to see every column.

In [ ]:
show(build_headline(df), HEADLINE_ID)

## 2. Showcase flight: JFK→LHR, 2020-02-09
Every forecast source's plan over the truth's 250 hPa wind speed, with the hindsight plan drawn as a wide translucent band beneath them. Use the dropdown to pick the lead and the slider to see each source's forecast winds instead.

In [ ]:
show(build_showcase_map(cfg), MAP_ID)

## 3. Regret vs lead time
Mean fuel regret by lead; the top row shows the forecast sources (IFS HRES, GraphCast, Pangu-Weather), eastbound and westbound flights side by side. The persistence baseline sits on its own row below, on its own scale because its regret is far larger. Lines are means and shaded bands are 90% CIs.

In [ ]:
show(build_lead_regret(df), LEAD_ID)

## 4. Wind error vs fuel regret
The RMS wind error along the hindsight plan's path against the plan's fuel regret, in two panels, forecast sources and the persistence baseline, each on its own scale. Faint points are single plans and lines are each source's binned median; plans above the plotted range are pinned at the top edge, with their true value in the hover.

In [ ]:
show(build_wind_error(df), WIND_ID)

## 5. Sanity panel
Checks that the experiment behaves physically: eastbound vs westbound replay times, the fuel-flow distribution, and how often an optimiser gap occurs and how large it is.

In [ ]:
show(build_sanity(df), SANITY_ID)

## 6. Stretch: geostrophic balance at 250 hPa
Is a forecast's wind near jet level balanced the way the real atmosphere is? The **ageostrophic fraction** is |V − V_g| / |V|, the share of the wind that the geostrophic wind V_g (from the geopotential gradient) does not explain, taken over 30–70°N and averaged over the experiment's flights.

Done as an opt-in add-on with a reduced method (sections 7 and 8): NeuralGCM and its ensemble. The WeatherBench 2 NeuralGCM stores start on 2020-01-01 and have 12-hourly steps, so the add-on keeps only the flights with a NeuralGCM initialisation at every lead and plans every source from steps 12 h apart.

In [ ]:
from regret.figures.geostrophic import FIG_ID as GEO_ID, build_geostrophic, geostrophic_summary
from IPython.display import Markdown

_geo_path = cfg.results_path.parent / "geostrophic.parquet"
if _geo_path.exists():
    _geo = pd.read_parquet(_geo_path)
    show(build_geostrophic(_geo), GEO_ID)
    display(Markdown(geostrophic_summary(_geo)))
else:
    display(Markdown(
        "*The geostrophic balance figure is an optional stretch and its result is not present. "
        "To produce it, run `uv run regret fetch --with geopotential` and then `uv run regret geostrophic`, "
        "then run the report again.*"))

## 7. NeuralGCM add-on: 5-source comparison
Reduced method: only the flights with a NeuralGCM initialisation at every lead (departures from 2020-01-06), and **every** source, not only NeuralGCM, planned from the steps at the lead and 12 h later (12-hourly planning), so differences between sources are not time-resolution differences. The add-on has its own candidate set (the truth-optimised plan and all five forecast sources) and its own results, separate from sections 1 to 5; the core experiment is unchanged. Lines are means and shaded bands are 90% CIs.

In [ ]:
from regret.figures.neuralgcm import FIG_ID as NGCM_ID, build_neuralgcm
from regret.figures.headline import SOURCE_LABELS as _LABELS

_ngcm_path = cfg.results_path.parent / "neuralgcm_regret.parquet"
_sens_path = cfg.results_path.parent / "neuralgcm_sensitivity.parquet"


def _ngcm_summary(addon, sens):
    """Where NeuralGCM sits (mean fuel regret, persistence excluded), and the 12-hourly planning effect."""
    fc = addon[~addon["is_truth_optimised"] & (addon["source"] != "persistence")]
    mean = fc.groupby(["source", "lead_h"])["regret_fuel_kg"].mean().unstack("lead_h")
    n = addon["flight_id"].nunique()
    leads = sorted(int(x) for x in mean.columns)
    short, long_ = leads[:2], leads[-1]
    out = [f"On {n} flights, NeuralGCM's mean fuel regret "]
    parts = []
    for label, ls in (("at the shortest leads", short), ("at the longest lead", [long_])):
        m = mean.loc[:, ls].mean(axis=1)
        rank = int(m.rank()["neuralgcm"])
        best = m.idxmin()
        txt = (f"{label} ({'-'.join(str(x) for x in (ls[0], ls[-1])) if len(ls) > 1 else ls[0]} h) is {m['neuralgcm']:.0f} kg, "
               f"rank {rank} of the {len(m)} forecast sources other than the persistence baseline")
        txt += (f", the lowest" if best == "neuralgcm" else
                f", {round(m['neuralgcm']) - round(m[best])} kg above {_LABELS.get(best, best)} ({m[best]:.0f} kg)")
        parts.append(txt)
    out.append("; ".join(parts) + ". ")
    if sens is not None:
        s = sens[sens["source"] != "persistence"]
        diff = (s["regret_12h_kg"] - s["regret_6h_kg"]).groupby(s["lead_h"]).mean()
        short_diff = float(diff.loc[[l for l in short if l in diff.index]].mean())
        names = [_LABELS.get(x, x) for x in dict.fromkeys(s["source"])]
        named = ", ".join(names[:-1]) + " and " + names[-1] if len(names) > 1 else names[0]
        out.append(
            f"Planning 12-hourly instead of 6-hourly adds roughly {short_diff:.0f} kg of mean regret at "
            f"{short[0]}-{short[-1]} h for {named} on the same flights, so short-lead numbers here "
            "mostly reflect time resolution, not forecast quality.")
    return "".join(out).strip()


if _ngcm_path.exists():
    _ngcm = pd.read_parquet(_ngcm_path)
    _sens = pd.read_parquet(_sens_path) if _sens_path.exists() else None
    show(build_neuralgcm(_ngcm), NGCM_ID)
    display(Markdown(_ngcm_summary(_ngcm, _sens)))
else:
    display(Markdown(
        "*The NeuralGCM add-on is optional and its result is not present. "
        "To produce it, run `uv run regret fetch --with neuralgcm,neuralgcm_ens`, `uv run regret neuralgcm` and `uv run regret ensemble`, "
        "then run the report again.*"))

## 8. Ensemble robust plan (NeuralGCM ensemble)
On the add-on's flights and 12-hourly planning, a plan is made on each of M members of the NeuralGCM ensemble, and each plan is replayed through every member. The **mean-robust** plan has the lowest mean cost across members and the **p90-robust** plan the lowest 90th-percentile cost; each chosen plan is then replayed through the truth. Regret is floored at 0 against the add-on's hindsight plan, and the share of flights where a chosen plan beats it is reported. M is a subset of the 50 members, chosen from measured run time (cost grows with the square of M). The IFS ensemble and GenCast are not used: WeatherBench 2 stores them only at 500/700/850 hPa.

In [ ]:
from regret.figures.ensemble import FIG_ID as ENS_ID, build_ensemble

_ens_path = cfg.results_path.parent / "ensemble_regret.parquet"


def _ens_summary(ens, addon):
    """M, mean-robust vs the best deterministic source at the longest lead, beats-hindsight, p90 note."""
    m = int(ens["members"].iloc[0])
    long_ = int(ens["lead_h"].max())
    at = ens[ens["lead_h"] == long_].groupby("choice")["regret_fuel_kg"].mean()
    det = addon[~addon["is_truth_optimised"] & (addon["source"] != "persistence") & (addon["lead_h"] == long_)]
    det = det.groupby("source")["regret_fuel_kg"].mean()
    best = det.idxmin()
    out = [f"The robust plans use M = {m} of the 50 members. At {long_} h the mean-robust plan's mean fuel regret is "
           f"{at['mean_robust']:.0f} kg against {det[best]:.0f} kg for the best deterministic source "
           f"({_LABELS.get(best, best)}). "]
    wins = int(ens["beats_hindsight"].sum())
    total = len(ens)
    if wins == 0:
        out.append(f"Neither choice ever beats the add-on hindsight plan (0 of {total} plans). ")
    else:
        out.append(f"A chosen plan beats the add-on hindsight plan in {wins} of {total} plans. ")
    if at["p90_robust"] > at["mean_robust"]:
        out.append(f"With M = {m} the 90th percentile is close to the maximum over members, so the p90-robust plan "
                   f"is overly cautious and costs more fuel (mean regret {at['p90_robust']:.0f} kg at {long_} h).")
    else:
        out.append(f"The p90-robust plan's mean regret at {long_} h is {at['p90_robust']:.0f} kg.")
    return "".join(out).strip()


if _ens_path.exists() and _ngcm_path.exists():
    _ens = pd.read_parquet(_ens_path)
    _addon = pd.read_parquet(_ngcm_path)
    show(build_ensemble(_ens, _addon), ENS_ID)
    display(Markdown(_ens_summary(_ens, _addon)))
else:
    display(Markdown(
        "*The ensemble robust plan is optional and its result is not present. "
        "To produce it, run `uv run regret fetch --with neuralgcm,neuralgcm_ens`, `uv run regret neuralgcm` and `uv run regret ensemble`, "
        "then run the report again.*"))

## 9. Method, assumptions and limitations

**Method.**
- Winds vary in space and time along the flight: bilinear in latitude/longitude, linear in time between 6-hourly steps. A forecast plan uses one initialisation and steps through its prediction steps as the flight progresses.
- The planner is a dynamic programme on a lateral grid aligned with the great circle (25 columns × 41 cross-track offsets spaced 60 km, at most ±3 offsets between columns). It keeps the earliest arrival at each node, at constant TAS (M0.85, FL350, ISA). With TAS and flight level fixed, the minimum-time plan is also the minimum-fuel and minimum-CO₂ plan.
- Replay flies a plan's fixed lateral path through the truth in steps of about 10 km, with OpenAP B789 fuel flow at the current mass; CO₂ = 3.16 × fuel.
- For each flight the candidates are the truth-optimised plan and every forecast plan. The lowest replayed cost is the hindsight plan, and a plan's **regret** is its replayed cost minus the hindsight plan's cost, for fuel, time and CO₂ separately.
- Wind error is the RMS of |V_forecast − V_truth| at 250 hPa along the hindsight plan's path.

### Assumptions and limitations
- Cruise only at fixed FL350 and M0.85; no climb, descent or step climbs.
- No ATC, North Atlantic Tracks, airspace or ETOPS constraints.
- OpenAP fuel models are accurate to roughly ±5–10%; regret is a difference, so most bias cancels.
- Winds at 1.5° under-resolve the jet core; ERA5 is treated as truth but is itself a model.
- Small sample: 2 months, 1 city pair, 1 aircraft. Results show spread, not significance; CIs are given.
- ISA temperature (no temperature effect on TAS); no turbulence.
- The hindsight plan is the best of the candidates, so the planner's grid limits what "best" means.
- The IFS ensemble is unused (no 250 hPa in WeatherBench 2), and so is GenCast; GFS is not in WeatherBench 2.

In [ ]:
print(f"elapsed_s={time.perf_counter() - _t0:.1f}")